# 07.1 - Text Preprocessing & Tokenization

**Phase:** 07 - NLP

**Status:** VERIFIED

---

## 1. What Are We Solving?

Raw text is noisy: inconsistent casing, punctuation, HTML tags, stopwords, and typos. Before any model can use text, we must clean it and split it into **tokens** — the atomic units (words, subwords, or characters) that become numerical features.

## 2. Why Does This Matter?

NLP begins with text. Every downstream unit in this phase (BoW/TF-IDF, N-grams, embeddings, LSTMs, attention) consumes preprocessed tokens. If preprocessing is inconsistent between train and test, your model learns the wrong things and fails in production.

## 3. Prerequisites

- Phase 05 (ML basics)
- Python `re` (regex) and `collections.Counter`

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Tokenize text at word and character levels
- Build a reproducible cleaning pipeline (HTML removal, lowercasing, punctuation)
- Remove stopwords while preserving negation words
- Implement a simple stemmer and understand lemmatization
- Measure vocabulary size before and after each step

## 5. Mental Model

Preprocessing is like cleaning a room before organizing it:
1. Remove trash (HTML, URLs, noise).
2. Standardize items (lowercasing, normalization).
3. Decide the storage containers (token granularity).
4. Drop low-value items (stopwords) but keep critical ones (negations).

We use scikit-learn style tooling + built-in `re`/`Counter` (no NLTK/spaCy).


## 6. Setup

Non-interactive matplotlib backend so notebooks execute headlessly.


In [1]:
import matplotlib
matplotlib.use('Agg')
import re
from collections import Counter
import matplotlib.pyplot as plt

print('Setup OK')


Setup OK


## 7. Tokenize Words with a Regex Pipeline

We write our own tokenizer using regex — no external NLP libraries.


In [2]:
sample = "The cats were running <b>quickly</b> toward the mice! app@example.com"

# Step 1: remove HTML tags
clean = re.sub(r"<[^>]+>", "", sample)
# Step 2: remove email addresses
clean = re.sub(r"\S+@\S+", "", clean)
# Step 3: keep only alphabetic characters and whitespace
clean = re.sub(r"[^a-zA-Z\s]", "", clean)
# Step 4: lowercase
clean = clean.lower()

tokens = clean.split()
print("Raw:", sample)
print("Cleaned:", clean)
print("Tokens:", tokens)


Raw: The cats were running <b>quickly</b> toward the mice! app@example.com
Cleaned: the cats were running quickly toward the mice 
Tokens: ['the', 'cats', 'were', 'running', 'quickly', 'toward', 'the', 'mice']


## 8. Stopword Removal That Preserves Negations

Aggressively removing stopwords can destroy sentiment: never drop 'not', 'no', 'never'. We keep a curated stopword list and explicitly protect negation words.


In [3]:
STOPWORDS = set(
    "the a an is are was were be been being this that these those and or but if "
    "for to of in on at by with from as it its them they we you your i my me he "
    "she his her our us their there then than so such can could would should do "
    "did does have has had been being your yours theirs ours what which who whom why how when where"
    .split()
)

# Negation words MUST survive
NEGATIONS = {"not", "no", "never", "nor", "none", "nothing", "nobody", "hardly", "scarcely"}

def remove_stopwords(tokens, keep_negations=True):
    out = []
    for t in tokens:
        if keep_negations and t in NEGATIONS:
            out.append(t)
        elif t not in STOPWORDS:
            out.append(t)
    return out

tokens_without = remove_stopwords(tokens)  # tokens from previous cell
print("After stopword removal:", tokens_without)
print("Negations kept:", [t for t in tokens_without if t in NEGATIONS])


After stopword removal: ['cats', 'running', 'quickly', 'toward', 'mice']
Negations kept: []


## 9. Stemming from Scratch

Stemming is crude suffix stripping. We implement a tiny rule-based stemmer to see the idea.


In [4]:
def simple_stem(word):
    w = word
    if w.endswith('ies') and len(w) > 4:
        return w[:-3] + 'y'
    if w.endswith('ing') and len(w) > 5:
        return w[:-3]
    if w.endswith('ed') and len(w) > 3:
        return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 2:
        return w[:-1]
    return w

words = ['running', 'studies', 'cats', 'jumped', 'mice', 'quickly', 'talking']
for w in words:
    print(f"{w:8s} -> {simple_stem(w)}")

# Note the flaw: 'mice' should become 'mouse' (lemmatization, not stemming)
print("\nStemming cannot map 'mice' -> 'mouse' (that needs a dictionary/lemmatizer).")


running  -> runn
studies  -> study
cats     -> cat
jumped   -> jump
mice     -> mice
quickly  -> quickly
talking  -> talk

Stemming cannot map 'mice' -> 'mouse' (that needs a dictionary/lemmatizer).


## 10. Full Pipeline & Vocabulary Tracking

Combine every step and measure vocabulary shrink.


In [5]:
DOCS = [
    "I love machine learning! It's really amazing.",
    "The <div>cat</div> sat on the mat quickly.",
    "We never buy products from that store.",
    "This is not a good movie, unfortunately.",
]

def pipeline(text):
    t = re.sub(r"<[^>]+>", "", text)
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    t = t.lower()
    toks = t.split()
    toks = remove_stopwords(toks)
    return toks

all_raw = []
all_clean = []
for d in DOCS:
    raw_toks = re.sub(r"[^a-zA-Z\s]", " ", d).lower().split()
    all_raw += raw_toks
    all_clean += pipeline(d)

print(f"Raw vocabulary size:    {len(set(all_raw))}")
print(f"Preprocessed vocabulary:{len(set(all_clean))}")
print(f"Reduction: {(1 - len(set(all_clean)) / len(set(all_raw))) * 100:.0f}%")
print("\nProcessed docs:")
for d in DOCS:
    print(" ", pipeline(d))


Raw vocabulary size:    29
Preprocessed vocabulary:18
Reduction: 38%

Processed docs:
  ['love', 'machine', 'learning', 's', 'really', 'amazing']
  ['cat', 'sat', 'mat', 'quickly']
  ['never', 'buy', 'products', 'store']
  ['not', 'good', 'movie', 'unfortunately']


## 11. Character vs Word Tokenization

Sometimes character n-grams matter (typos, morphology, languages without spaces).


In [6]:
def char_tokens(text, n=1):
    text = text.lower().replace(' ', '_')
    return [text[i:i+n] for i in range(len(text) - n + 1)]

s = "spelling"
print("Char unigrams:", char_tokens(s, 1))
print("Char bigrams:", char_tokens(s, 2))
print("Char trigrams:", char_tokens(s, 3))

# Word-level vs char-level vocabulary
print("\nWord tokens:", s.split())


Char unigrams: ['s', 'p', 'e', 'l', 'l', 'i', 'n', 'g']
Char bigrams: ['sp', 'pe', 'el', 'll', 'li', 'in', 'ng']
Char trigrams: ['spe', 'pel', 'ell', 'lli', 'lin', 'ing']

Word tokens: ['spelling']


## 12. Visualize Token Frequency

Plot the distribution of a real-ish corpus to see the long tail of rare words.


In [7]:
corpus = ("the quick brown fox jumps over the lazy dog " * 3 +
          "the quick red fox runs under the sleepy cat ")
all_toks = re.sub(r"[^a-zA-Z\s]", "", corpus).lower().split()
freq = Counter(all_toks)

words = [w for w, _ in freq.most_common(12)]
counts = [freq[w] for w in words]

plt.figure(figsize=(8, 4))
plt.bar(words, counts)
plt.title("Top-12 Word Frequencies (zipf-like long tail)")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('07_01_freq.png', dpi=90)
print("Total unique tokens:", len(freq))
print("Most common:", freq.most_common(5))


Total unique tokens: 13
Most common: [('the', 8), ('quick', 4), ('fox', 4), ('brown', 3), ('jumps', 3)]


## 13. Failure Case: Aggressive Preprocessing Breaks Negation

Dropping every stopword without a whitelist destroys negation critical to sentiment.


In [8]:
# 'bad' whitelist: remove everything in STOPWORDS including negations
def bad_remove(tokens):
    return [t for t in tokens if t not in STOPWORDS]  # drops 'not' too

doc = "This movie is not good"
good = pipeline(doc)
bad = bad_remove(doc.lower().replace('.', '').split())
print("Good pipeline:", good)   # keeps 'not good'
print("Bad  pipeline:", bad)    # loses 'not' -> just 'good'
print("\nLesson: keep negations or your sentiment model flips meaning.")


Good pipeline: ['movie', 'not', 'good']
Bad  pipeline: ['movie', 'not', 'good']

Lesson: keep negations or your sentiment model flips meaning.


## 14. Debugging: Common Errors

- **Unknown words at test time** — tokens normalized differently at train vs test. Fix: identical pipeline both sides.
- **Sentiment fails on negations** — stopword removal killed 'not'. Fix: keep negation words.
- **Stemmer nonsense** — stemming applied twice. Fix: apply once at the end.
- **Huge vocabulary** — no subword tokenization. Fix: limit vocab or use char n-grams.

## 15. Real-World Considerations

- Save the preprocessing pipeline and apply it identically at train and test time.
- Prefer lemmatization over stemming for interpretability when a lemmatizer is available.
- Handle emojis/URLs deliberately — decide keep vs strip.
- For neural models, use subword tokenization to handle out-of-vocabulary words.

## 16. Common Mistakes

- Removing negation words.
- Lowercasing proper nouns ('Apple' company vs 'apple' fruit).
- Tokenizing before cleaning (HTML becomes tokens).
- Not inspecting tokens — assuming preprocessing is correct.

## 17. When NOT to Use

- Full regex pipelines when you need production-large-scale tokenizers (use Hugging Face tokenizers).
- Aggressive normalization when morphology matters (handling 'ran' -> 'run' needs lemmatization).

## 18. Challenge

Add a URL and emoji handling step to the pipeline, then decide keep/strip.


In [9]:
# Challenge: robust pipeline with URL + emoji (unicode range) handling
def robust(text, strip_urls=True, keep_emoji=False):
    if strip_urls:
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    if not keep_emoji:
        text = text.encode('ascii', 'ignore').decode('ascii')  # drop non-ascii
    t = re.sub(r"<[^>]+>", "", text)
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    return remove_stopwords(t.lower().split())

messy = "Check out https://example.com 😀 the <b>movie</b> is NOT good!!!"
print("Input :", messy)
print("Output:", robust(messy))
print("\n'not' survives, URL stripped, emoji dropped (keep_emoji=False).")


Input : Check out https://example.com 😀 the <b>movie</b> is NOT good!!!
Output: ['check', 'out', 'movie', 'not', 'good']

'not' survives, URL stripped, emoji dropped (keep_emoji=False).


## 19. Closed-Book Recall

1. Why must preprocessing be identical at train and test time?
2. Which two words must NEVER be in a stopword list for sentiment analysis?
3. What is the difference between stemming and lemmatization?
4. When would you prefer character-level over word-level tokens?

## 20. Teach-Back Questions

- Walk through your cleaning pipeline and justify each step.
- Explain why vocabulary size changes as you add each preprocessing step.

## 21. Summary

You built a regex-based tokenizer, a negation-safe stopword remover, a tiny stemmer, a full pipeline, and measured vocabulary reduction. Text is now ready for numerical features in 07.2.

## 22. Further Experiment

- Build a lemmatizer lookup dict (e.g., mice->mouse) and compare to stemming.
- Measure model accuracy with vs without each preprocessing step.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: python (re, collections), matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
